# Baseline: vanilla PartCrafter (no restyle)

Plain `wgsxm/PartCrafter` image-to-3D, **no restyle** — the "theirs" baseline.

**Runtime → GPU (A100 / L4).**

`num_parts=1` gives a single-object mesh (fair baseline for the single-object prompts); bump it to decompose into parts. Flow: setup → load → run → download `baseline_glbs.zip`. Score locally with `eval/score_baseline.py`.

Inputs: zip of your images (`eval/dataset/images/`), filenames matching `captions.csv`.

In [ ]:
!nvidia-smi -L

In [ ]:
# 1. Clone PartCrafter + install (~10-15 min first time).
%cd /content
!git clone https://github.com/wgsxm/PartCrafter.git
%cd /content/PartCrafter
!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 \
    --index-url https://download.pytorch.org/whl/cu124
!pip install torch-cluster -f https://data.pyg.org/whl/torch-2.5.1+cu124.html
!pip install scikit-learn diffusers transformers einops 'huggingface_hub[hf_transfer]' \
    opencv-python-headless trimesh omegaconf scikit-image numpy==1.26.4 \
    peft jaxtyping typeguard matplotlib imageio imageio-ffmpeg pyrender colormaps \
    accelerate pillow

In [ ]:
# 2. Download weights + load pipeline.
import os, sys
os.environ['PYOPENGL_PLATFORM'] = 'egl'
%cd /content/PartCrafter
sys.path.insert(0, '/content/PartCrafter')
import torch
from huggingface_hub import snapshot_download
from src.pipelines.pipeline_partcrafter import PartCrafterPipeline
from src.models.briarmbg import BriaRMBG

snapshot_download('wgsxm/PartCrafter', local_dir='/content/weights/PartCrafter')
snapshot_download('briaai/RMBG-1.4', local_dir='/content/weights/RMBG-1.4')
rmbg_net = BriaRMBG.from_pretrained('/content/weights/RMBG-1.4').to('cuda').eval()
pipe = PartCrafterPipeline.from_pretrained('/content/weights/PartCrafter').to('cuda', torch.float16)
print('pipeline ready')

In [ ]:
# 3. Upload images.zip (locally: cd eval/dataset && zip -r images.zip images)
from google.colab import files
import zipfile, os, glob
os.makedirs('/content/inputs', exist_ok=True)
for name in files.upload():
    if name.endswith('.zip'):
        zipfile.ZipFile(name).extractall('/content/inputs')
imgs = sorted(glob.glob('/content/inputs/**/*.png', recursive=True) +
              glob.glob('/content/inputs/**/*.jpg', recursive=True))
print(f'{len(imgs)} images')

In [ ]:
# 4. Vanilla PartCrafter per input (NO restyle). Resumable.
import os, traceback, numpy as np, torch
from PIL import Image
from src.utils.data_utils import get_colored_mesh_composition
from src.utils.image_utils import prepare_image
OUT = '/content/baseline_glbs'; os.makedirs(OUT, exist_ok=True)
NUM_PARTS = 1               # 1 = single-object baseline; raise to decompose
NUM_TOKENS, STEPS, GUID = 1024, 50, 7.0

for i, p in enumerate(imgs, 1):
    stem = os.path.splitext(os.path.basename(p))[0]
    out_glb = f'{OUT}/{stem}.glb'
    if os.path.exists(out_glb): print(f'[{i}/{len(imgs)}] skip {stem}'); continue
    print(f'[{i}/{len(imgs)}] {stem} ...')
    try:
        pil = prepare_image(p, bg_color=np.array([1.0,1.0,1.0]), rmbg_net=rmbg_net)
        with torch.no_grad():
            outputs = pipe(
                image=[pil]*NUM_PARTS, attention_kwargs={'num_parts': NUM_PARTS},
                num_tokens=NUM_TOKENS,
                generator=torch.Generator(device=pipe.device).manual_seed(42),
                num_inference_steps=STEPS, guidance_scale=GUID,
                max_num_expanded_coords=int(1e9), use_flash_decoder=False).meshes
        import trimesh
        outputs = [m if m is not None else trimesh.Trimesh(vertices=[[0,0,0]], faces=[[0,0,0]])
                   for m in outputs]
        get_colored_mesh_composition(outputs).export(out_glb)
        print(f'    -> {out_glb}')
    except Exception:
        traceback.print_exc()

In [ ]:
# 5. Zip + download. Score locally:
#   .venv/bin/python eval/score_baseline.py eval/baseline_glbs_partcrafter eval/results_baseline_partcrafter.csv
import shutil
shutil.make_archive('/content/baseline_glbs', 'zip', '/content/baseline_glbs')
from google.colab import files
files.download('/content/baseline_glbs.zip')